## Note on approach history
An earlier version of this attempted deriving the full sidewalk network 
from polygon centerline extraction. Curb-cut driveway notches produced unavoidable 
spur artifacts even after spur-pruning, so this version pivots to Street Centerline 
as the base network while making centerlines of park paths. 

See the original attempt here:
[NYCPathways_v1_sidewalk_centerlines.ipynb](../archive/NYCPathways_v1_sidewalk_centerlines.ipynb)

Big Picture: I want to make a trip generation visualization tool to understand where are the points where people intersect intersect the most at. I want to ground this analysis so I will be basing them on two locations: Jackson Heights and Bed-Stuy. I will constrain this further by making a 20 (maybe 30) minute perimeter around each neighborhoods primary park (Herbert Von King and Travers). Additionally I will populate the points at which the analysis is being done based off of real census data.
The trip analysis will then be computed in Grasshopper. 

Intent: Find centerlines for NYC including parks in order to construct shortest trip generation
So far there is not a dataset including of centerlines including park paths, which is commonly used for making the shortest path.
This led me to NYC Planimetric Database: Sidewalk. database

Insert visualization

First I'm going to start to see what I have in the list (bc I can be forgetful) as there are some other plugins required for turning a polygon map into centerlines

In [1]:
!conda env list


# conda environments:
#
# * -> active
# + -> frozen
ZCDPGIS              *   /Users/temp/.conda/envs/ZCDPGIS
cdp312                   /Users/temp/.conda/envs/cdp312
                         /Users/temp/Documents/GitHub/cdp-mapping-systems/.conda
base                     /opt/anaconda3
                         /opt/miniconda3/envs/geo_env



In [2]:
!conda list | grep -E "geopandas|shapely|fiona|pyproj|matplotlib"

fiona                            1.10.1           py313h7df67bf_6      conda-forge
geopandas                        1.1.3            pypi_0               pypi
matplotlib-base                  3.10.9           py313h36cb854_0      conda-forge
matplotlib-inline                0.2.1            pypi_0               pypi
pyproj                           3.7.2            py313h6de5794_3      conda-forge
shapely                          2.1.2            py313h72d6987_0


In [3]:
!pip install centerline
!pip show folium
!pip install folium mapclassify

Name: folium
Version: 0.20.0
Summary: Make beautiful maps with Leaflet.js & Python
Home-page: https://github.com/python-visualization/folium
Author: Rob Story
Author-email: wrobstory@gmail.com
License: MIT
Location: /Users/temp/.conda/envs/ZCDPGIS/lib/python3.13/site-packages
Requires: branca, jinja2, numpy, requests, xyzservices
Required-by: cdptools, leafmap


In [4]:
!pip show centerline

Name: centerline
Version: 1.1.1
Summary: Calculate the centerline of a polygon
Home-page: https://github.com/fitodic/centerline
Author: Filip Todic
Author-email: todic.filip@gmail.com
License: MIT License
Location: /Users/temp/.conda/envs/ZCDPGIS/lib/python3.13/site-packages
Requires: Click, Fiona, numpy, scipy, Shapely
Required-by: 


In [5]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPolygon, LineString, Point
from shapely.ops import nearest_points
from centerline.geometry import Centerline

nta = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/2020_Neighborhood_Tabulation_Areas_(NTAs)_20260717.geojson")
street_centerline = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/Centerline_20260717-2.geojson")
open_space = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/NYC_Planimetric_Database__Sidewalk_20260717.geojson")

This section is defining all of the datasets to the same coordinate reference system.
To_crs is a command to do that.
"WORKING_CRS" is a variable holding the numered reference system

In [6]:
WORKING_CRS = 2263  # NAD83 State Plane Long Island (feet)

nta = nta.to_crs(WORKING_CRS)
street_centerline = street_centerline.to_crs(WORKING_CRS)
open_space = open_space.to_crs(WORKING_CRS)

This is grabbing only the "interior paths" which include park paths, housing projects paths, really anything thats not a primary sidewalk

In [7]:
interior_paths = open_space[open_space["sub_code"].astype(str) == "380010"]

This is filtering the parts of the dataset that I want and putting them into respected categories
Library Functions: {}, for, in, .items(), .str.contains(), gpd.GeoDataFrame(...)
For: "for every time (key, and name_filter) show up
In: do the following funciton


In [8]:
neighborhoods = {
    "bedstuy": "Bedford-Stuyvesant",
    "jacksonheights": "Jackson Heights"
}

boundaries = {}
for key, name_filter in neighborhoods.items():
    neighborhoods = {
    "bedstuy": "Bedford-Stuyvesant",
    "jacksonheights": "Jackson Heights"
}

boundaries = {}
for key, name_filter in neighborhoods.items():
    parts = nta[nta["ntaname"].str.contains(name_filter, na=False)]  # filter nta, specifically the ntaname column, to only include rows that contain the name_filter string. na=False excludes rows with NaN in ntaname.
    print(f"{key}: {len(parts)} NTA piece(s) found")
    boundaries[key] = gpd.GeoDataFrame(
        geometry=[parts.unary_union],
        crs=nta.crs
    )

interior_paths = open_space[open_space["sub_code"].astype(str) == "380010"]

study_area = gpd.GeoSeries(
    [b.geometry.iloc[0] for b in boundaries.values()],
    crs=nta.crs
).unary_union

interior_paths = interior_paths[interior_paths.intersects(study_area)].copy()
print(f"interior_paths narrowed to {len(interior_paths)} features within study area")

boundaries = {}
for key, name_filter in neighborhoods.items():
    parts = nta[nta["ntaname"].str.contains(name_filter, na=False)]  # filter nta, specifically the ntaname column, to only include rows that contain the name_filter string. na=False excludes rows with NaN in ntaname.
    print(f"{key}: {len(parts)} NTA piece(s) found")
    boundaries[key] = gpd.GeoDataFrame(
        geometry=[parts.unary_union],
        crs=nta.crs
    )

interior_paths = open_space[open_space["sub_code"].astype(str) == "380010"]

study_area = gpd.GeoSeries(
    [b.geometry.iloc[0] for b in boundaries.values()],
    crs=nta.crs
).unary_union

interior_paths = interior_paths[interior_paths.intersects(study_area)].copy()
print(f"interior_paths narrowed to {len(interior_paths)} features within study area")

bedstuy: 2 NTA piece(s) found
jacksonheights: 1 NTA piece(s) found
interior_paths narrowed to 106 features within study area
bedstuy: 2 NTA piece(s) found
jacksonheights: 1 NTA piece(s) found
interior_paths narrowed to 106 features within study area


/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3995240496.py:18: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[parts.unary_union],
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3995240496.py:18: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[parts.unary_union],
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3995240496.py:27: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ).unary_union
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3995240496.py:37: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[parts.unary_union],
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3995240496.py:37: DeprecationWarning: The 'unary_union' attribute is deprecated, use th

In [9]:
neighborhoods = {
    "bedstuy": "Bedford-Stuyvesant",
    "jacksonheights": "Jackson Heights"
}

boundaries = {}
for key, name_filter in neighborhoods.items():
    parts = nta[nta["ntaname"].str.contains(name_filter, na=False)] #saying filter the nta, specifically the ntaname column, to only include rows that contain the name_filter string (e.g., "Bedford-Stuyvesant" or "Jackson Heights"). The na=False argument ensures that any rows with NaN values in the ntaname column are excluded from the filter.
    print(f"{key}: {len(parts)} NTA piece(s) found")
    boundaries[key] = gpd.GeoDataFrame(
        geometry=[parts.unary_union],
        crs=nta.crs
    )

# Step 3 — filter to Interior Sidewalk (unchanged, still runs first — cheap attribute filter)
interior_paths = open_space[open_space["sub_code"].astype(str) == "380010"]

# Step 4 — boundaries (unchanged)
neighborhoods = {
    "bedstuy": "Bedford-Stuyvesant",
    "jacksonheights": "Jackson Heights"
}
boundaries = {}
for key, name_filter in neighborhoods.items():
    parts = nta[nta["ntaname"].str.contains(name_filter, na=False)]
    print(f"{key}: {len(parts)} NTA piece(s) found")
    boundaries[key] = gpd.GeoDataFrame(geometry=[parts.unary_union], crs=nta.crs)

# NEW — limit interior_paths to just the selected neighborhoods, combined
study_area = gpd.GeoSeries(
    [b.geometry.iloc[0] for b in boundaries.values()],
    crs=nta.crs
).unary_union

interior_paths = interior_paths[interior_paths.intersects(study_area)].copy()
print(f"interior_paths narrowed to {len(interior_paths)} features within study area")

bedstuy: 2 NTA piece(s) found
jacksonheights: 1 NTA piece(s) found
bedstuy: 2 NTA piece(s) found
jacksonheights: 1 NTA piece(s) found
interior_paths narrowed to 106 features within study area


/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3473573026.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[parts.unary_union],
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3473573026.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[parts.unary_union],
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3473573026.py:27: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundaries[key] = gpd.GeoDataFrame(geometry=[parts.unary_union], crs=nta.crs)
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/3473573026.py:27: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  boundaries[key] = gpd.GeoDataFrame(geometry=[parts.unary_union], crs=nta.crs)
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r

To write whats occurring

In [10]:
def get_dangling_endpoints(lines_gdf):
    exploded = lines_gdf.explode(index_parts=False).reset_index(drop=True)
    endpoint_counts = {}
    coords_list = []
    for geom in exploded.geometry:
        start, end = geom.coords[0], geom.coords[-1]
        coords_list.append((start, end))
        for pt in [start, end]:
            key = (round(pt[0], 1), round(pt[1], 1))
            endpoint_counts[key] = endpoint_counts.get(key, 0) + 1

    dangling = []
    for start, end in coords_list:
        for pt in [start, end]:
            key = (round(pt[0], 1), round(pt[1], 1))
            if endpoint_counts[key] == 1:
                dangling.append(Point(pt))
    return dangling

def connect_to_street(dangling_points, street_union, max_distance=65):
    connectors = []
    for pt in dangling_points:
        nearest_on_street = nearest_points(pt, street_union)[1]
        distance = pt.distance(nearest_on_street)
        if distance <= max_distance:
            connectors.append(LineString([pt, nearest_on_street]))
        # else: too far from any street — likely an interior dead-end, skip it
    return connectors

In [11]:
def build_network(boundary_gdf, neighborhood_name):
    street_clipped = gpd.clip(street_centerline, boundary_gdf)
    street_clipped = street_clipped[["geometry"]].copy()
    street_clipped["segment_type"] = "street"

    park_clipped = gpd.clip(interior_paths, boundary_gdf)
    park_line_geoms = []
    for geom in park_clipped.geometry:
        polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
        for poly in polys:
            try:
                park_line_geoms.append(Centerline(poly).geometry)
            except Exception:
                pass

    park_lines = gpd.GeoDataFrame(geometry=park_line_geoms, crs=park_clipped.crs)
    park_lines = prune_spurs(park_lines, min_length=15)
    park_lines["segment_type"] = "park_path"

    dangling_points = get_dangling_endpoints(park_lines)
    street_union = street_clipped.geometry.unary_union
    connectors = connect_to_street(dangling_points, street_union)
    connectors_gdf = gpd.GeoDataFrame(geometry=connectors, crs=park_lines.crs)
    connectors_gdf["segment_type"] = "park_connector"

    result = gpd.GeoDataFrame(
        pd.concat([
            street_clipped[["geometry", "segment_type"]],
            park_lines[["geometry", "segment_type"]],
            connectors_gdf[["geometry", "segment_type"]]
        ], ignore_index=True),
        crs=street_clipped.crs
    )
    result["neighborhood"] = neighborhood_name
    return result

In [12]:
def prune_spurs(lines_gdf, min_length=15, passes=3):
    lines = lines_gdf.explode(index_parts=False).reset_index(drop=True)
    for _ in range(passes):
        endpoint_counts = {}
        for geom in lines.geometry:
            for pt in [geom.coords[0], geom.coords[-1]]:
                key = (round(pt[0], 1), round(pt[1], 1))
                endpoint_counts[key] = endpoint_counts.get(key, 0) + 1

        def is_spur(geom):
            start = (round(geom.coords[0][0], 1), round(geom.coords[0][1], 1))
            end = (round(geom.coords[-1][0], 1), round(geom.coords[-1][1], 1))
            dangling = endpoint_counts[start] == 1 or endpoint_counts[end] == 1
            return dangling and geom.length < min_length

        lines = lines[~lines.geometry.apply(is_spur)].reset_index(drop=True)
    return lines

In [13]:
all_networks = [build_network(boundary, name) for name, boundary in boundaries.items()]
final_network = pd.concat(all_networks, ignore_index=True).reset_index(drop=True)
final_network["segment_id"] = final_network.index

import os
os.makedirs("outputs", exist_ok=True)
final_network.to_crs(epsg=4326).to_file("outputs/walk_network.geojson", driver="GeoJSON")

/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/388543726.py:21: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  street_union = street_clipped.geometry.unary_union


/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_72634/388543726.py:21: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  street_union = street_clipped.geometry.unary_union


## Exporting per-neighborhood centerlines for Grasshopper

The citywide `street_centerline` (before it gets merged with park paths into `walk_network`) is too large to import into Grasshopper reliably — export it clipped to each neighborhood boundary separately instead, so each file only covers the area actually needed.

In [ ]:
for key, boundary in boundaries.items():
    clipped = gpd.clip(street_centerline, boundary)
    clipped.to_crs(4326).to_file(f"outputs/centerline_{key}.geojson", driver="GeoJSON")
    print(f"{key}: {len(clipped)} segments -> outputs/centerline_{key}.geojson")